## Merge Triangle Mesh objects

This example shows how to select any number of existing triangle mesh objects from an Evo workspace using checkboxes, merge them into a single mesh, and publish the result as a new triangle mesh object using the Evo Python SDK.

A [triangle mesh](https://developer.seequent.com/docs/data-structures/geoscience-objects/schemas/objects/triangle-mesh/) stores its geometry in two tables:
- `triangles.vertices`: the `x`, `y`, `z` coordinates of every vertex.
- `triangles.indices`: triplets of 0-based indices (`n0`, `n1`, `n2`) into the vertex table, one row per triangle.

To merge the meshes, their vertex tables are concatenated, and the triangle indices of each mesh are offset by the total number of vertices in the meshes before it so that they still point at the correct vertices.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameters must be provided:

- The client ID of your Evo application.
- The callback/redirect URL of your Evo application.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

The selected workspace must contain at least two triangle mesh objects. You can publish one using the [Publish a Triangular Mesh](../../publish-triangular-mesh/publish-triangular-mesh.ipynb) notebook.

In [ ]:
from evo.notebooks import ServiceManagerWidget

# Evo app credentials
client_id = "<your-client-id>"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

manager = await ServiceManagerWidget.with_auth_code(client_id=client_id, redirect_url=redirect_url).login()

### Use the Evo Python SDK to create an object client and a data client

The data client uses its own cache under the temp directory for Parquet files.

In [ ]:
import os
import tempfile

from evo.common.utils import Cache
from evo.objects import ObjectAPIClient

# The object client will manage your auth token and Geoscience Object API requests.
object_client = ObjectAPIClient(manager.get_environment(), manager.get_connector())

parquet_cache = Cache(os.path.join(tempfile.gettempdir(), "evo-notebook-data"), mkdir=True)
print(f"Caching Parquet data in: {parquet_cache.root}")

# The data client will manage downloading and saving your data as Parquet and publishing your data to Evo storage.
data_client = object_client.get_data_client(parquet_cache)

### Create the triangle mesh helper

[triangle_mesh_helpers.py](triangle_mesh_helpers.py) contains the helpers used in this notebook:
- `TriangleMeshHelper` lists the triangle meshes in the workspace, downloads mesh data, preserves the coordinate reference system, and displays a link to view the new object in the Evo portal.
- `MeshSelectorWidget` displays a checkbox for each mesh, plus fields for the name and path of the merged mesh.

In [ ]:
from triangle_mesh_helpers import TriangleMeshHelper

mesh_helper = TriangleMeshHelper(object_client, data_client)

### Select the triangle meshes to merge

List all triangle mesh objects in the workspace and tick the checkboxes of the meshes to merge (at least two). Use **Select all** to toggle every mesh at once. You can also set the name and path of the merged object that will be published.

In [ ]:
from triangle_mesh_helpers import MeshSelectorWidget

triangle_mesh_objects = await mesh_helper.list_triangle_meshes()
if len(triangle_mesh_objects) < 2:
    raise RuntimeError(
        f"Found {len(triangle_mesh_objects)} triangle mesh object(s) in the workspace, but at least two are required. "
        "Publish a triangle mesh using the 'publish-triangular-mesh' notebook."
    )

print(f"Found {len(triangle_mesh_objects)} triangle mesh objects.")
mesh_selector = MeshSelectorWidget(triangle_mesh_objects)
mesh_selector

### Download and merge the selected meshes

Run the cell below after making your selections. All selected meshes must use the same coordinate reference system; this is checked before any mesh data is downloaded. The vertices of all selected meshes are concatenated and the triangle indices of each mesh are offset by the total number of vertices in the meshes before it.

> **Note:** Only the mesh geometry is merged. Attributes, parts and edges of the source meshes are not carried over to the merged mesh.

In [ ]:
import numpy as np

selected_meshes = mesh_selector.selected_meshes
if len(selected_meshes) < 2:
    raise ValueError(f"Select at least two triangle meshes to merge ({len(selected_meshes)} selected).")

# Download only the object definitions first, so the coordinate reference systems can be checked before any data is downloaded.
mesh_objects = []
meshes_by_crs = []  # (crs, [mesh paths]) for each distinct CRS, including unspecified.
for mesh in selected_meshes:
    mesh_object = await mesh_helper.download_triangle_mesh_object(mesh)
    mesh_objects.append(mesh_object)

    crs = mesh_object.get("coordinate_reference_system") or "unspecified"
    crs_group = next((paths for group_crs, paths in meshes_by_crs if group_crs == crs), None)
    if crs_group is None:
        crs_group = []
        meshes_by_crs.append((crs, crs_group))
    crs_group.append(mesh.path)

# All meshes must share the same coordinate reference system, and an unspecified CRS does not match a specified one.
# This notebook does not reproject data.
if len(meshes_by_crs) > 1:
    crs_details = "\n".join(
        f"  - {mesh_helper.describe_crs(crs)}: " + ", ".join(f"'{path}'" for path in paths)
        for crs, paths in meshes_by_crs
    )
    raise ValueError(
        f"Cannot merge the selected meshes because they use {len(meshes_by_crs)} different coordinate "
        f"reference systems:\n{crs_details}\n"
        "Deselect the meshes that do not match, or reproject them to a common coordinate reference system "
        "and run this cell again."
    )
merged_crs = meshes_by_crs[0][0]

vertex_arrays, index_arrays = [], []
mesh_summaries = []
vertex_offset = 0
for mesh, mesh_object in zip(selected_meshes, mesh_objects):
    vertices, indices = await mesh_helper.download_triangle_mesh_data(mesh, mesh_object)

    vertex_arrays.append(vertices)
    # Offset the indices so they point at this mesh's vertices within the merged vertex table.
    index_arrays.append(indices + np.uint64(vertex_offset))
    vertex_offset += len(vertices)
    mesh_summaries.append((len(vertices), len(indices), mesh.path))

merged_vertices = np.vstack(vertex_arrays)
merged_indices = np.vstack(index_arrays)

print(f"{'Vertices':>12} {'Triangles':>12}  Mesh")
for vertex_count, triangle_count, mesh_path in mesh_summaries:
    print(f"{vertex_count:>12,} {triangle_count:>12,}  {mesh_path}")
print(f"{len(merged_vertices):>12,} {len(merged_indices):>12,}  Merged mesh ({len(selected_meshes)} meshes)")

### Save the merged vertices and indices as Parquet

The vertices are saved using the `float-array-3` table format and the triangle indices using the `index-array-3` table format, as required by the triangle mesh schema.

In [ ]:
import pyarrow as pa
from evo_schemas.components import BoundingBox_V1_0_1, Triangles_V1_2_0_Indices, Triangles_V1_2_0_Vertices

from evo.objects.utils.table_formats import FLOAT_ARRAY_3, INDEX_ARRAY_3

vertices_table = pa.table(
    {
        "x": pa.array(merged_vertices[:, 0], type=pa.float64()),
        "y": pa.array(merged_vertices[:, 1], type=pa.float64()),
        "z": pa.array(merged_vertices[:, 2], type=pa.float64()),
    }
)
indices_table = pa.table(
    {
        "n0": pa.array(merged_indices[:, 0], type=pa.uint64()),
        "n1": pa.array(merged_indices[:, 1], type=pa.uint64()),
        "n2": pa.array(merged_indices[:, 2], type=pa.uint64()),
    }
)

mesh_vertices = Triangles_V1_2_0_Vertices.from_dict(data_client.save_table(vertices_table, table_format=FLOAT_ARRAY_3))
mesh_indices = Triangles_V1_2_0_Indices.from_dict(data_client.save_table(indices_table, table_format=INDEX_ARRAY_3))

# Define the bounding box from the merged vertices.
min_x, min_y, min_z = merged_vertices.min(axis=0)
max_x, max_y, max_z = merged_vertices.max(axis=0)
merged_bounding_box = BoundingBox_V1_0_1(
    min_x=float(min_x),
    max_x=float(max_x),
    min_y=float(min_y),
    max_y=float(max_y),
    min_z=float(min_z),
    max_z=float(max_z),
)

### Create the merged triangle mesh and publish it to Evo

Geoscience object data must conform to a specific object schema. The `evo-schemas` package provides Pydantic models that make it easy to work with the equivalent JSON schemas. For this example we'll use v2.2.0 of the triangle mesh schema, via the relevant Pydantic model.

In [ ]:
import os
from datetime import datetime, timezone

from evo_schemas.components import Triangles_V1_2_0
from evo_schemas.objects import TriangleMesh_V2_2_0

from evo.notebooks import FeedbackWidget

object_name = mesh_selector.merged_name.strip() or "Merged_Mesh_SDK_demo"
if object_name.endswith(".json"):
    object_name = object_name[: -len(".json")]
if os.environ.get("CI"):
    object_name += f"_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')}"
object_path = mesh_selector.merged_path.strip().strip("/")

# The object name requires the `.json` extension.
full_obj_path = f"{object_path}/{object_name}.json" if object_path else f"{object_name}.json"

merged_mesh = TriangleMesh_V2_2_0(
    name=object_name,
    uuid=None,
    description=f"Merge of {len(selected_meshes)} triangle meshes: "
    + ", ".join(f"'{mesh.path}'" for mesh in selected_meshes)
    + ".",
    bounding_box=merged_bounding_box,
    tags={"Source": "Jupyter Notebook"},
    coordinate_reference_system=mesh_helper.to_crs_model(merged_crs),
    triangles=Triangles_V1_2_0(vertices=mesh_vertices, indices=mesh_indices),
)

# Upload the Parquet data to Evo.
await data_client.upload_referenced_data(merged_mesh.as_dict(), FeedbackWidget("Uploading data"))

# Create the geoscience object.
merged_mesh_metadata = await object_client.create_geoscience_object(full_obj_path, merged_mesh.as_dict())
print(f"Published merged triangle mesh to '{full_obj_path}' (Object ID: {merged_mesh_metadata.id})")

### View the object in the Evo portal

In [ ]:
mesh_helper.build_portal_url(merged_mesh_metadata)

Success! You now have a new geoscience object in Evo containing the merged mesh data.

## Summary

In this example, we've completed the following:
* Listed the triangle mesh objects in a workspace and selected the meshes to merge using checkboxes.
* Downloaded the vertex and triangle index data of each selected mesh.
* Merged the meshes by concatenating the vertices and offsetting the triangle indices of each mesh by the number of preceding vertices.
* Saved the merged data in Parquet format and combined it into the triangle-mesh schema format.
* Uploaded the Parquet files and the newly assembled object in JSON format to Evo.